In [22]:
import sys
sys.path.append("/root/FoodSeg_mask2former")

import os
import json
from tqdm import tqdm
from gradio_app.model_inference_classification2 import predict_masks, vit_food_classification_local

# Set paths
input_dir = "FoodSeg_mask2former/test_images"
output_json_path = "FoodSeg_mask2former/batch_results.json"
output_vis_dir = "FoodSeg_mask2former/visualized_outputs"

# Create directory to save overlay images if it doesn't exist
os.makedirs(output_vis_dir, exist_ok=True)

# Store results
all_results = []

# Loop over all image files
for filename in tqdm(os.listdir(input_dir)):
    if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    image_path = os.path.join(input_dir, filename)

    try:
        # Run both models
        vis_image, detected_labels, pixel_percentages, total_energy = predict_masks(image_path)
        top_label = vit_food_classification_local(image_path)

        # Save visualized output
        vis_save_path = os.path.join(output_vis_dir, f"vis_{filename}")
        vis_image.save(vis_save_path)

        # Collect result
        result = {
            "image": filename,
            "vit_prediction": top_label,
            "detected_labels": detected_labels,
            "pixel_percentages": pixel_percentages,
            "total_energy_kcal": round(total_energy, 2),
            "vis_path": vis_save_path
        }
        all_results.append(result)

    except Exception as e:
        print(f"[Error] Failed processing {filename}: {e}")

# Save all results to a JSON file
with open(output_json_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n✅ Done! Results saved to: {output_json_path}")


100%|██████████| 101/101 [18:39<00:00, 11.08s/it]


✅ Done! Results saved to: FoodSeg_mask2former/batch_results.json
